In [6]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "allritz2021chimpanzees")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "wavering_chimpanzees_leipzig_2021.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [7]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_csv(complete_path_1)


df['study_id']="allritz2021chimpanzees"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)



In [8]:

df.rename(columns={"subject": "participant"}, inplace=True)

comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
df['participant'] = df['participant'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df['participant'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
df= df.merge(apedf,left_on='participant', right_on='name', how='left')

df['participant'].unique()

array(['alex', 'jahaga', 'kofi'], dtype=object)

In [9]:
age_list = [['alex',13],
            ['jahaga',22],
            ['kofi',9]]
for x,y in age_list:
    df.loc[df.participant == x, ['age_in_years']] = y
# df.columns

df.rename(columns={"distance": "symbolic_distance"}, inplace=True)

In [10]:


studyID_standardized=df[['study_id', 'participant','age_in_years','sex', 'species',
    'session', 'trial','trial_subset', 'subset', 
       'early_item', 'late_item', 'coord_early_x', 'coord_early_y',
       'coord_late_x', 'coord_late_y', 'magnitude', 'symbolic_distance', 'accuracy',
       'latency', 'wavering', 'wavering_coder2' 
        ]]
comp_out_path_stand = os.path.join(out_pathway, 'allritz2021chimpanzees_standardized.csv')
studyID_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)


names =studyID_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
studyID_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'allritz2021chimpanzees_glossary.csv')
studyID_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
